In [ ]:
import time

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
from unsloth import FastLanguageModel
    
# RTX 5070 Ti optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

[unsloth.import_fixes|WARNING]Unsloth: torch==2.12.0.dev20260223+cu128 requires torchvision>=0.27.0, but found torchvision==0.26.0.dev20260223+cu128. Please refer to https://pytorch.org/get-started/previous-versions/ for more information.
Detected a pre-release build. Continuing with a warning. Set UNSLOTH_SKIP_TORCHVISION_CHECK=1 to silence this.
/tmp/ipykernel_82432/2743889244.py:5: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
WARNING 03-01 00:50:25 [__init__.py:144] xccl is not enabled in this torch build, communication is not available.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-135M",
    torch_dtype=torch.bfloat16,  # native bf16 instead of fp32
    device_map="auto",
    attn_implementation="sdpa",  # fused scaled dot-product attention (auto-dispatches to flash kernel)
)
model = torch.compile(model, mode="max-autotune")  # tries more kernel variants than reduce-overhead

In [ ]:
# count tokens per sec
with torch.inference_mode():  # faster than no_grad
    input_ids = tokenizer("John: Hello, how are you?", return_tensors="pt").input_ids.to(device)

    # warmup pass for torch.compile (first call compiles the graph)
    _ = model.generate(input_ids, max_new_tokens=10, do_sample=True, temperature=0.7)

    start_time = time.time()
    output = model.generate(input_ids, max_new_tokens=2048, do_sample=True, temperature=0.7)
    end_time = time.time()

    tokens_generated = output.shape[1] - input_ids.shape[1]
    time_taken = end_time - start_time
    tokens_per_second = tokens_generated / time_taken

    print(f"Tokens generated: {tokens_generated}")
    print(f"Time taken: {time_taken:.2f} seconds")
    print(f"Tokens per second: {tokens_per_second:.2f}")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-270m-it",
    max_seq_length=2048,
    load_in_4bit=False,   # no quantization — model is tiny, quant adds overhead
    load_in_8bit=False,   # ^^
    dtype=torch.bfloat16, # native bf16
    fast_inference=True,
)
text_streamer = TextStreamer(tokenizer)

In [ ]:
# count tokens per sec
with torch.inference_mode():
    input_ids = tokenizer("John: Hello, how are you?", return_tensors="pt").input_ids.to(device)

    # warmup for any JIT compilation
    _ = model.generate(input_ids, max_new_tokens=10, do_sample=True, temperature=0.7)

    start_time = time.time()
    output = model.generate(input_ids, streamer=text_streamer, max_new_tokens=2048, do_sample=True, temperature=0.7)
    end_time = time.time()

    tokens_generated = output.shape[1] - input_ids.shape[1]
    time_taken = end_time - start_time
    tokens_per_second = tokens_generated / time_taken

    print(f"Tokens generated: {tokens_generated}")
    print(f"Time taken: {time_taken:.2f} seconds")
    print(f"Tokens per second: {tokens_per_second:.2f}")

In [3]:
# --- vLLM with Unsloth model ---
from vllm import LLM, SamplingParams

llm = LLM(model="unsloth/gemma-3-270m-it", dtype="bfloat16")
params = SamplingParams(temperature=0.7, max_tokens=2048)

start_time = time.time()
outputs = llm.generate(["John: Hello, how are you?"], params)
end_time = time.time()

for output in outputs:
    generated_text = output.outputs[0].text
    tokens = len(output.outputs[0].token_ids)
    tps = tokens / (end_time - start_time)
    print(generated_text)
    print(f"\nTokens: {tokens}, Time: {end_time - start_time:.2f}s, Tok/s: {tps:.2f}")

INFO 03-01 00:50:29 [utils.py:250] non-default args: {'dtype': 'bfloat16', 'disable_log_stats': True, 'model': 'unsloth/gemma-3-270m-it'}
INFO 03-01 00:50:29 [model.py:530] Resolved architecture: Gemma3ForCausalLM
INFO 03-01 00:50:29 [model.py:1540] Using max model len 32768
INFO 03-01 00:50:30 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-01 00:50:30 [vllm.py:666] Asynchronous scheduling is enabled.
WARNING 03-01 00:50:31 [system_utils.py:140] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
WARNING 03-01 00:50:33 [__init__.py:144] xccl is not enabled in this torch build, communication is not available.


Skipping import of cpp extensions due to incompatible torch version 2.12.0.dev20260223+cu128 for torchao version 0.17.0.dev20260228+cu128             Please see https://github.com/pytorch/ao/issues/2919 for more info


(EngineCore_DP0 pid=82707) INFO 03-01 00:50:36 [core.py:96] Initializing a V1 LLM engine (v0.16.0rc1.dev161+g52ee21021.d20260223) with config: model='unsloth/gemma-3-270m-it', speculative_config=None, tokenizer='unsloth/gemma-3-270m-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_d

(EngineCore_DP0 pid=82707) <frozen importlib._bootstrap_external>:1325: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore_DP0 pid=82707) <frozen importlib._bootstrap_external>:1325: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(EngineCore_DP0 pid=82707) INFO 03-01 00:50:38 [weight_utils.py:567] No model.safetensors.index.json found in remote.
(EngineCore_DP0 pid=82707) INFO 03-01 00:50:38 [default_loader.py:291] Loading weights took 0.10 seconds


...

(EngineCore_DP0 pid=82707) 2026-03-01 00:50:42,676 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore_DP0 pid=82707) 2026-03-01 00:50:42,686 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:00<00:00, 63.47it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 54.85it/s]


(EngineCore_DP0 pid=82707) INFO 03-01 00:50:44 [gpu_model_runner.py:5227] Graph capturing finished in 2 secs, took 0.06 GiB
(EngineCore_DP0 pid=82707) INFO 03-01 00:50:44 [core.py:272] init engine (profile, create kv cache, warmup model) took 5.80 seconds
(EngineCore_DP0 pid=82707) INFO 03-01 00:50:46 [vllm.py:666] Asynchronous scheduling is enabled.
INFO 03-01 00:50:46 [llm.py:349] Supported tasks: ['generate']


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


John: I'm doing well, thank you for asking.
John: I'm fine, just doing my daily tasks.
John: I'm enjoying my time here.
John: I'm glad to be here.
John: I'm looking forward to meeting you.
John: I'm sure we'll have a great time.
John: I'm ready for anything.
John: I am.
John: Hello, how are you?
John: I'm doing well, thank you for asking.
John: I'm fine, just doing my daily tasks.
John: I'm enjoying my time here.
John: I'm glad to be here.
John: I'm looking forward to meeting you.
John: I'm sure we'll have a great time.
John: I'm ready for anything.
John: I am.
John: Hello, how are you?
John: I'm doing well, thank you for asking.
John: I'm fine, just doing my daily tasks.
John: I'm enjoying my time here.
John: I'm glad to be here.
John: I'm looking forward to meeting you.
John: I'm sure we'll have a great time.
John: I'm ready for anything.
John: I am.
John: Hello, how are you?
John: I'm doing well, thank you for asking.
John: I'm fine, just doing my daily tasks.
John: I'm enjoying my

In [4]:
start_time = time.time()
outputs = llm.generate(["John: Hello, how are you?"], params)
end_time = time.time()

for output in outputs:
    generated_text = output.outputs[0].text
    tokens = len(output.outputs[0].token_ids)
    tps = tokens / (end_time - start_time)
    print(generated_text)
    print(f"\nTokens: {tokens}, Time: {end_time - start_time:.2f}s, Tok/s: {tps:.2f}")


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


John: I'm fine, thank you. I'm happy to help.

Maria: Hello, how are you?
Maria: I'm fine, thank you. I'm happy to help.

John: How are you?
John: I'm doing well, thank you. I'm just a little tired.

Maria: Hello, how are you?
Maria: I'm fine, thank you. I'm happy to help.

John: How are you?
John: I'm up to my eyeballs.

Maria: Hello, how are you?
Maria: I'm fine, thank you. I'm happy to help.

John: How are you?
John: I'm doing well, thank you. I'm just a little tired.

Maria: Hello, how are you?
Maria: I'm fine, thank you. I'm happy to help.

John: How are you?
John: I'm doing well, thank you. I'm just a little tired.

Maria: Hello, how are you?
Maria: I'm fine, thank you. I'm happy to help.

John: How are you?
John: I'm doing well, thank you. I'm just a little tired.

Maria: Hello, how are you?
Maria: I'm fine, thank you. I'm happy to help.

John: How are you?
John: I'm doing well, thank you. I'm just a little tired.

Maria: Hello, how are you?
Maria: I'm fine, thank you. I'm happ